# TS convergence, one structure: manual

`refine_temperature_bracket_manually` from `phase_diagram_workflows.free_energies.ts_convergence.single`.

A temperature-scaling (TS) run over a temperature bracket is only trustworthy if its forward and backward sweeps agree. The disagreement is the **criterion**; if it is above the tolerance the bracket is narrowed and run again.

In this mode **you** are the loop: every call looks at the bracket log on disk and either reports what it found, or submits the next bracket and returns at once. It never blocks and never chains by itself. This is the most transparent driver and the one to use when you want to look at every bracket before going on.

| driver | who submits the next bracket | notebook |
|---|---|---|
| `refine_temperature_bracket_manually` | you, by calling again | this one |
| `refine_temperature_bracket_with_chain` | everything up front, executor resolves the dependencies | `chain.ipynb` |
| `refine_temperature_bracket_with_resubmission` | every bracket job submits the next one | `resubmission.ipynb` |

Everything is the same small Al case (`../_common.py`): 32 atoms, 100 + 100 steps, so a bracket takes seconds. Not production-quality TI.

In [1]:
import sys
from pathlib import Path

# _common.py (shared settings) sits two levels up from this notebook
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_common.py").is_file())))
import _common

_common.setup_environment()


In [2]:
import os
import time

import pandas as pd
from executorlib import SingleNodeExecutor

from phase_diagram_workflows.free_energies.ts_convergence.single import (
    load_bracket_history,
    refine_temperature_bracket_manually,
)

## 1. Inputs

In [3]:
structure = _common.al_structure()                 # 32 atoms
potential_df = _common.mishin_al_potential()
calphy_parameters = _common.calphy_parameters()
INITIAL_BRACKET = _common.INITIAL_BRACKET          # (700, 720) K
print(f"{len(structure)} atoms, bracket {INITIAL_BRACKET}")


32 atoms, bracket (700.0, 720.0)


## 2. Call it until it settles

`tolerance=-1` can never be met (the criterion is an absolute value), so the first bracket is always narrowed: we see a `"resubmitted"` and not just `"pending"` -> `"converged"`. `step_upper=5` lowers the upper bound by 5 K per narrowing, `(700, 720)` -> `(700, 715)`.

The returned `status` says what to do next: `"pending"` (running, call again later), `"incomplete"` (the folder was read mid-write, also just call again), `"resubmitted"` (did not converge, a narrower bracket was submitted), `"converged"`.

In [4]:
root = str(_common.results_directory("single_manual"))
executor = SingleNodeExecutor(hostname_localhost=True, max_cores=1, cache_directory=os.path.join(root, "executorlib_cache"))

result = {"status": "pending"}
start = time.time()
while result["status"] in ("pending", "incomplete") and time.time() - start < 600:
    result = refine_temperature_bracket_manually(
        input_structure=structure,
        calphy_parameters=calphy_parameters,
        potential_df=potential_df,
        executor=executor,
        working_directory_root=root,
        initial_bracket=INITIAL_BRACKET,
        tolerance=-1.0,        # unreachable: forces a narrowing step
        step_upper=5.0,
    )
    print(f"[{time.time() - start:5.0f} s] {result['status']:12s} {result['bracket']}")
    if result["status"] in ("pending", "incomplete"):
        time.sleep(5)

executor.shutdown(wait=False, cancel_futures=False)
assert result["status"] == "resubmitted", result

[    0 s] pending      (700.0, 720.0)


[    5 s] incomplete   (700.0, 720.0)


[   10 s] incomplete   (700.0, 720.0)


[   15 s] incomplete   (700.0, 720.0)


Input validation successful. Proceeding with calculation.


[   20 s] resubmitted  (700.0, 715.0)


## 3. The bracket log

A plain CSV, one row per bracket, written by the driver. It, and not the executor's cache, is the record of what was tried, so it can be read from any session. The criterion of the narrower bracket is still empty: it was submitted, not finished.

In [5]:
pd.read_csv(os.path.join(root, "bracket_log.csv"))

,t_low,t_high,criterion
0,700.0,720.0,0.032157
1,700.0,715.0,NaN


## 4. Calling again picks up where it left off

The next call finds the narrower bracket in the log. Wait for it by calling again; once it has run, the criterion appears in the log. Nothing is recomputed for the first bracket.

In [6]:
executor = SingleNodeExecutor(hostname_localhost=True, max_cores=1, cache_directory=os.path.join(root, "executorlib_cache"))
result = {"status": "pending"}
start = time.time()
while result["status"] in ("pending", "incomplete") and time.time() - start < 600:
    result = refine_temperature_bracket_manually(
        input_structure=structure, calphy_parameters=calphy_parameters, potential_df=potential_df,
        executor=executor, working_directory_root=root, initial_bracket=INITIAL_BRACKET,
        tolerance=-1.0, step_upper=5.0,
    )
    print(f"[{time.time() - start:5.0f} s] {result['status']:12s} {result['bracket']}")
    if result["status"] in ("pending", "incomplete"):
        time.sleep(5)
executor.shutdown(wait=False, cancel_futures=False)

tried_brackets, criteria_by_bracket = load_bracket_history(root)
print("tried:", sorted(tried_brackets))
print("criteria:", criteria_by_bracket)
assert (700.0, 715.0) in tried_brackets and criteria_by_bracket[(700.0, 715.0)] is not None

[    0 s] pending      (700.0, 715.0)


[    5 s] incomplete   (700.0, 715.0)


[   10 s] incomplete   (700.0, 715.0)


[   15 s] incomplete   (700.0, 715.0)


Input validation successful. Proceeding with calculation.


[   20 s] resubmitted  (700.0, 710.0)
tried: [(700.0, 710.0), (700.0, 715.0), (700.0, 720.0)]
criteria: {(700.0, 720.0): 0.0321569854476102, (700.0, 715.0): 0.00933143843806}
